# 법인카드 정산 자동화 · 이상거래 탐지 — 전처리 (Preprocessing)

**목적**: `법인카드_이상거래_EDA_v3.ipynb`에서 확인한 데이터 품질 이슈·데이터 누수 문제·Feature Engineering 결과를 반영해,
`train`/`test` 원본 CSV를 모델 학습에 바로 투입 가능한 형태로 정리한다.

**이 노트북에서 지키는 원칙 (모두 EDA_v3 근거)**
- **Look-ahead 피처는 만들지 않는다.** EDA 9절의 `사용자평균사용액`/`가맹점평균금액`/`거래금액_Zscore`(카드·가맹점 전체 기간 통계)는
  진단용으로만 유효했고, 실제 파이프라인에는 9-1절의 **Expanding Window(시점 이전 데이터만 사용)** 버전만 사용한다.
- **콜드스타트 정보를 숨기지 않는다.** EDA 10-4절에서 확인했듯 `fillna(median)`으로 결측을 채우면 "이력 없음" 상태가
  "평범한 값"으로 위장된다. 이 노트북은 결측을 임의로 채우지 않고 `카드/가맹점 첫거래여부` 플래그로 남겨, 트리 기반
  모델(LightGBM/XGBoost 등, EDA 10절 권장)이 결측을 네이티브로 처리하도록 둔다.
- **`'_'` 플레이스홀더는 명시적으로 NaN 처리한다.** EDA 3절에서 확인했듯 `.isnull()`로 잡히지 않는 은닉 결측치다.
- **금액=0원, 전월 실적 음수 같은 특이값은 제거하지 않고 플래그로 남긴다.** EDA 3절 결론(0원 거래는 위험 신호 아님,
  전월 실적 음수는 오히려 위험 신호 후보)에 따라 임의 삭제 대신 플래그 피처로 판단을 모델에 넘긴다.

**⚠️ train/test 분할 방식에 대한 중요한 사실 확인**
`.data/train`(173.6만 건)과 `.data/test`(21.7만 건)는 **같은 모집단의 무작위 분할**이다(둘 다 2021-01-01~2024-12-31 동일 기간,
이상거래 비율도 3.687% vs 3.687%로 사실상 동일, test에 등장하는 카드 2,577개는 **전부** train에도 존재).
즉 **미래 시점 홀드아웃이 아니라 같은 시기의 무작위 샘플**이다. 이 상태에서 시간 기반 피처(최근7일사용횟수, 카드누적사용액,
Expanding Window 통계)를 train/test 각각 따로 계산하면, test에 있는 카드의 과거 거래가 train 파일에 들어있어도 test
계산에서는 안 보이므로 실제로는 이력이 있는 카드를 "첫 거래"로 잘못 취급하게 된다. 그래서 이 노트북은 **train+test를
합쳐서 시간순으로 시간 기반 피처를 계산한 뒤, 원래 소속(train/test)으로 다시 나눈다.** 이 결합은 각 피처가 여전히
"해당 거래 시점 이전 정보만" 사용하도록(shift(1)/rolling) 만드는 것이므로 데이터 누수가 아니라, 오히려 원래대로 각자
계산했을 때 생기는 인위적인 콜드스타트 오분류를 없애는 작업이다.

## 1. 라이브러리 & 데이터 로드

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import warnings
import json

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

RAW_DIR = '../.data'
TRAIN_DIR = os.path.join(RAW_DIR, 'train')
TEST_DIR = os.path.join(RAW_DIR, 'test')
FILE_PATTERN = '21-2_카드데이터_*.csv'
OUT_DIR = os.path.join(RAW_DIR, 'processed')
os.makedirs(OUT_DIR, exist_ok=True)


def load_transaction_data(data_dir, pattern=FILE_PATTERN):
    '''data_dir 내 csv 파일을 자동 인식하여 하나의 DataFrame으로 통합. (EDA_v3 2절과 동일한 로더)'''
    files = sorted(glob.glob(os.path.join(data_dir, pattern)))
    if not files:
        raise FileNotFoundError(f"'{data_dir}' 경로에서 '{pattern}' 패턴의 csv 파일을 찾지 못했습니다.")
    frames = [pd.read_csv(f, encoding='utf-8', low_memory=False) for f in files]
    return pd.concat(frames, ignore_index=True)


train_raw = load_transaction_data(TRAIN_DIR)
test_raw = load_transaction_data(TEST_DIR)
print(f"train_raw: {train_raw.shape}")
print(f"test_raw : {test_raw.shape}")

train_raw: (1735885, 32)
test_raw : (216986, 32)


In [2]:
# train/test가 정말 같은 모집단의 무작위 분할인지 재확인 (위 인트로에서 언급한 근거를 이 노트북 안에서도 재현)
print(f"train 이상거래 비율: {train_raw['이상거래여부'].mean()*100:.3f}%")
print(f"test  이상거래 비율: {test_raw['이상거래여부'].mean()*100:.3f}%")

train_cards = set(train_raw['카드KEY'])
test_cards = set(test_raw['카드KEY'])
card_overlap = len(test_cards & train_cards)
print(f"\ntest 카드 {len(test_cards):,}개 중 train에도 존재하는 카드: {card_overlap:,}개 "
      f"({card_overlap/len(test_cards)*100:.1f}%)")
print(f"train 기간: {train_raw['승인일자'].min()} ~ {train_raw['승인일자'].max()}")
print(f"test  기간: {test_raw['승인일자'].min()} ~ {test_raw['승인일자'].max()}")

train 이상거래 비율: 3.687%
test  이상거래 비율: 3.687%

test 카드 2,577개 중 train에도 존재하는 카드: 2,577개 (100.0%)
train 기간: 20210101 ~ 20241231
test  기간: 20210101 ~ 20241231


In [3]:
# train/test 원래 소속을 태깅한 뒤, 시간 기반 피처를 함께 계산하기 위해 결합한다.
train_raw['데이터셋구분'] = 'train'
test_raw['데이터셋구분'] = 'test'
df = pd.concat([train_raw, test_raw], ignore_index=True)
print("결합 데이터 shape:", df.shape)

결합 데이터 shape: (1952871, 33)


## 2. 결측치 정규화 (`'_'` → NaN)

EDA_v3 3절에서 확인했듯 `남녀구분코드`, `개인법인구분코드_가맹점`, `가맹점형태구분코드`, `일시불할부구분코드`,
`가맹점승인업종코드`, `승인거래코드`, `승인발생경로코드` 등 문자열 컬럼에는 결측치(NaN)와 별도로 `'_'` 플레이스홀더가
쓰인다. `.isnull()`로는 잡히지 않으므로 명시적으로 NaN 처리한다.

In [4]:
underscore_cols = [
    col for col in df.columns
    if pd.api.types.is_object_dtype(df[col]) and (df[col] == '_').any()
]
print("'_' 플레이스홀더가 있는 컬럼:", underscore_cols)

before_na = df[underscore_cols].isna().sum()
df[underscore_cols] = df[underscore_cols].replace('_', np.nan)
after_na = df[underscore_cols].isna().sum()

pd.DataFrame({'처리 전 결측': before_na, "'_' → NaN 처리 후 결측": after_na})

'_' 플레이스홀더가 있는 컬럼: ['남녀구분코드', '승인거래코드', '승인발생경로코드', '가맹점승인업종코드', '개인법인구분코드_가맹점', '가맹점형태구분코드', '일시불할부구분코드']


,처리 전 결측,'_' → NaN 처리 후 결측
남녀구분코드,380,190664
승인거래코드,0,970
승인발생경로코드,0,897
가맹점승인업종코드,0,2385
개인법인구분코드_가맹점,29176,211187
가맹점형태구분코드,12265,117890
일시불할부구분코드,0,11489


## 3. 날짜/시간 파생 변수 (EDA_v3 7·9절과 동일한 정의)

In [5]:
df['거래일자'] = pd.to_datetime(df['승인일자'], format='%Y%m%d')
df['거래연월'] = df['거래일자'].dt.to_period('M')
df['거래요일_한글'] = df['거래일자'].dt.dayofweek.map(
    {0: '월', 1: '화', 2: '수', 3: '목', 4: '금', 5: '토', 6: '일'}
)


def categorize_hour(h):
    if 0 <= h < 6:
        return '심야(00-05)'
    elif 6 <= h < 12:
        return '오전(06-11)'
    elif 12 <= h < 18:
        return '오후(12-17)'
    else:
        return '저녁(18-23)'


df['시간대구간'] = df['승인시간대'].apply(categorize_hour)
df['월말여부'] = df['거래일자'].dt.day >= 25

df[['거래일자', '거래연월', '거래요일_한글', '시간대구간', '월말여부']].head()

,거래일자,거래연월,거래요일_한글,시간대구간,월말여부
0,2021-02-07,2021-02,일,오전(06-11),False
1,2021-02-19,2021-02,금,오후(12-17),False
2,2021-01-11,2021-01,월,오후(12-17),False
3,2021-03-28,2021-03,일,오후(12-17),True
4,2021-03-05,2021-03,금,오후(12-17),False


## 4. 이상값·특이값 플래그 (EDA_v3 3절 결론 반영)

셋 다 **행을 삭제하지 않고** 플래그 컬럼으로만 남긴다 — EDA에서 확인했듯 셋 다 단순 오류로 단정할 근거가 부족하거나
(0원 거래는 오히려 이상거래 비율이 평균보다 낮음), 오히려 위험 신호일 수 있어서(전월 실적 음수) 삭제가 정보 손실이다.

In [6]:
# (1) 통합승인금액 = 0원: EDA 3절에서 이상거래 비율 0.52%(전체 평균 3.69%)로 오히려 낮음을 확인 → 취소/인증성 거래로 추정
df['취소성거래_추정'] = (df['통합승인금액'] == 0).astype(int)

# (2) 전월 실적 음수: EDA 3절에서 이상거래 비율이 평균의 최대 3배 이상으로 높음을 확인 → 위험 신호 후보 플래그로 보존
df['전월매출건수_음수여부'] = (df['전월_매출건수'] < 0).astype(int)
df['전월매출금액_음수여부'] = (df['전월_매출금액'] < 0).astype(int)

# (3) 카드이용한도금액: EDA 3절에서 법인카드 거래의 약 98%가 0원(미기재)으로 기록됨을 확인 → 값 자체보다 "미기재 추정" 여부가 더 신뢰할 수 있는 신호
df['법인카드_한도미기재추정'] = (
    (df['개인법인구분코드_회원'] == 2.0) & (df['카드이용한도금액'] == 0)
).astype(int)
df['카드이용한도금액_구간'] = pd.Categorical(
    df['카드이용한도금액'],
    categories=[0, 1_000_000, 5_000_000, 10_000_000],
    ordered=True,
)

flag_cols = ['취소성거래_추정', '전월매출건수_음수여부', '전월매출금액_음수여부', '법인카드_한도미기재추정']
(df[flag_cols].mean() * 100).round(3).rename('비율(%)')

취소성거래_추정        0.147
전월매출건수_음수여부     0.025
전월매출금액_음수여부     0.054
법인카드_한도미기재추정    9.514
Name: 비율(%), dtype: float64

## 5. 시점 안전(Leak-free) Feature Engineering — EDA_v3 9절·9-1절·10-4절 근거

`최근7일사용횟수`·`카드누적사용액`은 원래도 시점 안전(rolling/cumsum, 해당 거래까지의 정보만 사용)했으므로 그대로 가져온다.
`사용자평균사용액`·`가맹점평균금액`·`거래금액_Zscore`는 EDA 9절에서 전체 기간(look-ahead) 버전으로 만들었다가 9-1절에서
데이터 누수를 확인했으므로, **이 노트북에서는 Expanding Window(shift(1)) 버전만** 만든다. 첫 거래(과거 이력 없음)는
EDA 10-4절 교훈대로 `fillna`로 채우지 않고 `카드/가맹점 첫거래여부` 플래그와 결측(NaN)으로 그대로 남긴다.

In [7]:
# 카드/가맹점별 시간순 정렬. 승인SEQ는 파일(분기)별로 채번되어 train/test 결합 후에는 완전히 유일하지 않을 수 있으므로
# 데이터셋구분까지 정렬 키에 포함해 그나마 결정론적으로 순서를 고정한다(실제로는 (카드KEY,승인일자,승인SEQ) 조합이
# train 내에서 이미 유일함을 확인했다 — 완전한 동시성 해소는 원본에 타임스탬프가 없어 불가능).
df = df.sort_values(['카드KEY', '거래일자', '승인SEQ', '데이터셋구분'])

# --- 시점 안전 롤링/누적 피처 (EDA_v3 9절과 동일한 방식, 원래도 leak-free) ---
# groupby+rolling 결과의 인덱스는 (카드KEY, 거래일자) 멀티인덱스라 원래 행 순서로 바로 대입할 수 없으므로,
# 정렬 전 위치(orig_positions)를 저장해뒀다가 위치 기반으로 안전하게 복원한다.

def add_rolling_count(frame, key_col, date_col, window='7D', out_col='최근7일사용횟수'):
    t = frame[[key_col, date_col]].copy()
    t['_one'] = 1
    t_sorted = t.sort_values([key_col, date_col])
    orig_positions = t_sorted.index
    t_sorted = t_sorted.set_index(date_col)
    rolled = t_sorted.groupby(key_col)['_one'].rolling(window).sum()
    result = pd.Series(rolled.values, index=orig_positions).sort_index()
    frame[out_col] = result
    return frame


df = add_rolling_count(df, '카드KEY', '거래일자')
df['카드누적사용액'] = df.groupby('카드KEY')['통합승인금액'].cumsum()

# --- Expanding Window(leak-free) 피처 (EDA 9-1절) ---
card_amt = df.groupby('카드KEY')['통합승인금액']
df['사용자평균사용액_확장'] = card_amt.transform(lambda s: s.expanding().mean().shift(1))
df['사용자표준편차_확장'] = card_amt.transform(lambda s: s.expanding().std().shift(1))

merchant_amt = df.groupby('가맹점KEY')['통합승인금액']
df['가맹점평균금액_확장'] = merchant_amt.transform(lambda s: s.expanding().mean().shift(1))

df['거래금액_Zscore_확장'] = (
    (df['통합승인금액'] - df['사용자평균사용액_확장'])
    / df['사용자표준편차_확장'].replace(0, np.nan)
)

# --- 콜드스타트 플래그 (EDA 10-4절 교훈: 결측을 채우지 말고 명시적으로 남긴다) ---
df['카드첫거래여부'] = df['사용자평균사용액_확장'].isna().astype(int)
df['가맹점첫거래여부'] = df['가맹점평균금액_확장'].isna().astype(int)

df = df.sort_index()  # concat 직후의 원래 행 순서(=train 다음 test) 복원

ew_cols = ['최근7일사용횟수', '카드누적사용액', '사용자평균사용액_확장', '가맹점평균금액_확장', '거래금액_Zscore_확장']
flag_cols2 = ['카드첫거래여부', '가맹점첫거래여부']
print("시점 안전 피처 결측 비율(%):")
print((df[ew_cols].isna().mean() * 100).round(2))
print("\n콜드스타트 플래그 비율(%):")
print((df[flag_cols2].mean() * 100).round(2))

시점 안전 피처 결측 비율(%):
최근7일사용횟수          0.00
카드누적사용액           0.00
사용자평균사용액_확장       0.13
가맹점평균금액_확장        0.51
거래금액_Zscore_확장    0.34
dtype: float64

콜드스타트 플래그 비율(%):
카드첫거래여부     0.13
가맹점첫거래여부    0.51
dtype: float64


## 6. 범주형 변수 dtype 정리

EDA_v3 10절 결론(트리 기반 앙상블 권장)에 맞춰 원-핫 인코딩 대신 `category` dtype으로만 캐스팅한다.
LightGBM은 `category` dtype을 네이티브로 처리하고, XGBoost/scikit-learn 계열을 쓸 경우 `col.cat.codes`로
정수 인코딩하거나 `pd.get_dummies`를 나중에 적용하면 된다. 결측(NaN)은 임의로 채우지 않고 그대로 category의
"결측"으로 남긴다 — EDA 3절에서 확인했듯 `남녀구분코드`가 NaN(구 `'_'`)이라는 사실 자체가 법인카드 신호였기 때문이다.

In [8]:
CATEGORICAL_COLS = [
    '개인법인구분코드_회원', '국내해외여부', '남녀구분코드', '승인거래코드', '승인발생경로코드',
    '개인법인구분코드_가맹점', '가맹점여부_신규', '인터넷판매여부', '가맹점상태코드', '가맹점형태구분코드',
    '로고구분코드', '일시불할부구분코드', '카드구분코드', '가맹점누적매출금액_구간화', '연령',
    '가맹점광역시도코드', '가맹점승인업종코드', '시간대구간', '거래요일_한글',
]

for col in CATEGORICAL_COLS:
    df[col] = df[col].astype('category')

df[CATEGORICAL_COLS].dtypes

개인법인구분코드_회원      category
국내해외여부           category
남녀구분코드           category
승인거래코드           category
승인발생경로코드         category
개인법인구분코드_가맹점     category
가맹점여부_신규         category
인터넷판매여부          category
가맹점상태코드          category
가맹점형태구분코드        category
로고구분코드           category
일시불할부구분코드        category
카드구분코드           category
가맹점누적매출금액_구간화    category
연령               category
가맹점광역시도코드        category
가맹점승인업종코드        category
시간대구간            category
거래요일_한글          category
dtype: object

## 7. train/test 재분리 및 정합성 검증

In [9]:
train_df = df[df['데이터셋구분'] == 'train'].drop(columns=['데이터셋구분']).reset_index(drop=True)
test_df = df[df['데이터셋구분'] == 'test'].drop(columns=['데이터셋구분']).reset_index(drop=True)

print(f"train_df: {train_df.shape}  (원본 train_raw: {train_raw.shape[0]:,}행)")
print(f"test_df : {test_df.shape}  (원본 test_raw : {test_raw.shape[0]:,}행)")
assert len(train_df) == len(train_raw) and len(test_df) == len(test_raw), "재분리 후 행 수가 원본과 다릅니다."

print(f"\ntrain 이상거래 비율: {train_df['이상거래여부'].mean()*100:.3f}%")
print(f"test  이상거래 비율: {test_df['이상거래여부'].mean()*100:.3f}%")

print("\ntrain 콜드스타트 비율(%):", (train_df[['카드첫거래여부', '가맹점첫거래여부']].mean() * 100).round(2).to_dict())
print("test  콜드스타트 비율(%):", (test_df[['카드첫거래여부', '가맹점첫거래여부']].mean() * 100).round(2).to_dict())

train_df: (1735885, 50)  (원본 train_raw: 1,735,885행)
test_df : (216986, 50)  (원본 test_raw : 216,986행)

train 이상거래 비율: 3.687%
test  이상거래 비율: 3.687%

train 콜드스타트 비율(%): {'카드첫거래여부': 0.13, '가맹점첫거래여부': 0.51}
test  콜드스타트 비율(%): {'카드첫거래여부': 0.14, '가맹점첫거래여부': 0.5}


## 8. 모델링에 사용할 피처 목록 정리 — 프로덕션 스키마 기준 4단계 Tier

원본 컬럼은 하나도 삭제하지 않았다. 대신 "실제 운영 환경(기술명세서 §3.1)에서 이 피처를 계산할 수 있는 근거가
어디서 오는가"를 기준으로 4단계 Tier로 나눈다. 실제 `transactions` 테이블은 `(merchant, amount, ts, card_id,
raw_payload JSONB)`뿐이라 이 AI Hub 데이터의 32개 컬럼이 그대로 오는 게 아니기 때문이다.

| Tier | 근거 | 프로덕션 가용성 |
|---|---|---|
| **Tier 0** | `transactions` 핵심 필드(가맹점·금액·시각·카드ID)만으로 계산 | 거의 확실히 유지됨 |
| **Tier 1** | 가맹점 속성 원본 그대로. 실제로는 `merchant_categories` 캐시(카카오/웹검색, §7-1)로 대체될 자리이며 **스키마 자체가 다르다** | 값은 못 쓰지만 "가맹점 속성이 필요하다"는 사실은 유지 |
| **Tier 2** | 사용자·부서·직급 등 조직 데이터 | 이 데이터셋엔 아예 없음 — `users`/`teams` 조인이 실제로 연동돼야 채워짐 |
| **Tier 3** | 카드사 정산/명세 데이터 특유 필드(전월 매출, 카드 한도, 인적정보 등) | `raw_payload`에 포함될지 미확정(§11.2 Open Issue) |

**사용 원칙**: `raw_payload` 스키마가 확정되기 전까지는 모델을 **Tier 0(+연동 완료 시 Tier 1)만으로** 먼저 만들고,
Tier 3는 "확보되면 붙이는 보강 피처"로 취급한다. 아래 `select_features()`로 어떤 조합을 쓸지 명시적으로 고른다.

In [10]:
ID_COLS = ['카드KEY', '가맹점KEY', '승인SEQ']
LABEL_COLS = ['이상거래여부', '이상거래유형', '이상거래설명']  # EDA 10-3절: 카드사 라벨이지 정산담당자 라벨이 아님 — 참고용/타깃 용도로만
RAW_DATE_COLS = ['기준년월', '승인일자']  # 파생 변수(거래일자 등)로 대체 가능

# --- Tier 0: transactions 핵심 필드(가맹점KEY/카드KEY/금액/시각)만으로 계산 가능 ---
TIER0_TRANSACTION_SAFE = [
    '승인시간대', '통합승인금액',
    '거래일자', '거래연월', '거래요일_한글', '시간대구간', '월말여부', '취소성거래_추정',
    '최근7일사용횟수', '카드누적사용액',
    '사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장', '카드첫거래여부',
    '가맹점평균금액_확장', '가맹점첫거래여부',
]

# --- Tier 1: 가맹점 속성 원본. 실제로는 merchant_categories 캐시(§7-1)로 대체될 자리이며
#     그쪽은 {industry_code, industry_label, confidence, source} 스키마라 아래 컬럼과 1:1로 안 맞는다.
#     즉 "이 값 자체"가 아니라 "가맹점 속성이 신호로 쓸모 있다"는 사실만 이 노트북에서 확인된 것이다.
TIER1_MERCHANT_ENRICHMENT = [
    '가맹점광역시도코드', '가맹점승인업종코드', '가맹점형태구분코드', '가맹점상태코드',
    '가맹점여부_신규', '가맹점누적매출금액_구간화', '인터넷판매여부',
]

# --- Tier 2: 사용자·부서·직급 등 조직 데이터. 이 데이터셋엔 없다 — users/teams 조인이 실제로 붙어야 채워짐 ---
TIER2_ORG_CONTEXT = []  # placeholder: 예) 부서, 직급, 팀 평균 대비 지출 배수 등 (EDA "추가 확보하면 좋은 데이터")

# --- Tier 3: 카드사 정산/명세 데이터 특유 필드. raw_payload 포함 여부 미확정(§11.2) ---
TIER3_ISSUER_OPTIONAL = [
    '개인법인구분코드_회원', '국내해외여부', '카드이용한도금액', '카드이용한도금액_구간', '법인카드_한도미기재추정',
    '연령', '남녀구분코드', '승인거래코드', '승인발생경로코드', '개인법인구분코드_가맹점', '로고구분코드',
    '일시불할부구분코드', '카드구분코드', '할부가능개월수', '전월_매출건수', '전월_매출금액',
    '전월매출건수_음수여부', '전월매출금액_음수여부', '경과일수_최종이용일자',
]

FEATURE_TIERS = {
    'tier0_transaction_safe': TIER0_TRANSACTION_SAFE,
    'tier1_merchant_enrichment': TIER1_MERCHANT_ENRICHMENT,
    'tier2_org_context': TIER2_ORG_CONTEXT,
    'tier3_issuer_optional': TIER3_ISSUER_OPTIONAL,
}

feature_cols = sum(FEATURE_TIERS.values(), [])

# 기존에 컬럼 기준(제외 목록 방식)으로 뽑았던 피처 집합과 정확히 같은지 검증 — 하나도 빠뜨리거나 중복하지 않았는지 확인
_legacy_feature_cols = [c for c in train_df.columns if c not in ID_COLS + LABEL_COLS + RAW_DATE_COLS + ['fold']]
assert set(feature_cols) == set(_legacy_feature_cols), (
    set(_legacy_feature_cols) ^ set(feature_cols)
)
assert len(feature_cols) == len(set(feature_cols)), "Tier 간 중복된 컬럼이 있습니다."

for tier_name, cols in FEATURE_TIERS.items():
    print(f"[{tier_name}] {len(cols)}개")
    print(cols)
    print()

print(f"전체 추천 피처: {len(feature_cols)}개 (Tier 0~3 합)")

[tier0_transaction_safe] 16개
['승인시간대', '통합승인금액', '거래일자', '거래연월', '거래요일_한글', '시간대구간', '월말여부', '취소성거래_추정', '최근7일사용횟수', '카드누적사용액', '사용자평균사용액_확장', '사용자표준편차_확장', '거래금액_Zscore_확장', '카드첫거래여부', '가맹점평균금액_확장', '가맹점첫거래여부']

[tier1_merchant_enrichment] 7개
['가맹점광역시도코드', '가맹점승인업종코드', '가맹점형태구분코드', '가맹점상태코드', '가맹점여부_신규', '가맹점누적매출금액_구간화', '인터넷판매여부']

[tier2_org_context] 0개
[]

[tier3_issuer_optional] 19개
['개인법인구분코드_회원', '국내해외여부', '카드이용한도금액', '카드이용한도금액_구간', '법인카드_한도미기재추정', '연령', '남녀구분코드', '승인거래코드', '승인발생경로코드', '개인법인구분코드_가맹점', '로고구분코드', '일시불할부구분코드', '카드구분코드', '할부가능개월수', '전월_매출건수', '전월_매출금액', '전월매출건수_음수여부', '전월매출금액_음수여부', '경과일수_최종이용일자']

전체 추천 피처: 42개 (Tier 0~3 합)


In [11]:
def select_features(tiers):
    '''사용할 Tier 이름들을 받아 피처 목록을 반환한다.

    raw_payload 스키마가 확정되기 전까지는 select_features(['tier0_transaction_safe'])처럼
    Tier 0(+ merchant_categories 연동이 끝났다면 tier1)만 쓰는 모델을 먼저 만드는 것을 권장한다.
    '''
    cols = []
    for t in tiers:
        cols += FEATURE_TIERS[t]
    return cols


prod_minimal_feats = select_features(['tier0_transaction_safe'])
prod_with_merchant_feats = select_features(['tier0_transaction_safe', 'tier1_merchant_enrichment'])
benchmark_feats = select_features(list(FEATURE_TIERS.keys()))

print(f"프로덕션 최소(Tier 0)만: {len(prod_minimal_feats)}개")
print(f"Tier 0 + 가맹점 연동(Tier 1): {len(prod_with_merchant_feats)}개")
print(f"AI Hub 벤치마크 전체(Tier 0~3): {len(benchmark_feats)}개")

프로덕션 최소(Tier 0)만: 16개
Tier 0 + 가맹점 연동(Tier 1): 23개
AI Hub 벤치마크 전체(Tier 0~3): 42개



## 9. Train 내부 검증 분할 (StratifiedGroupKFold, 카드 단위)

`test_df`는 최종 평가 전까지 손대지 않는다. 하이퍼파라미터 튜닝·조기종료·룰 임계값 선택(EDA 8절)을 위해
`train_df` 안에서 별도 검증 fold를 만든다.

- **카드(`카드KEY`) 단위로 그룹화**한다. EDA_v3 11절에서 카드가 분기를 넘어 평균 92.3% 재사용됨을 확인했고, 이 노트북의
  Expanding Window 피처(`사용자평균사용액_확장` 등)는 카드별 과거 이력을 담고 있다. 같은 카드의 거래가 fold-train과
  fold-valid에 걸쳐 있으면 "그 카드의 평소 패턴"이 양쪽에 새어 들어가 검증 성능이 낙관적으로 부풀 수 있으므로,
  한 카드의 모든 거래는 반드시 같은 fold에 속하게 한다.
- **`이상거래여부`로 층화(stratify)**한다. 클래스 불균형이 약 26:1이라 단순 GroupKFold만 쓰면 fold마다 이상거래
  비율이 들쭉날쭉할 수 있다. `StratifiedGroupKFold`는 그룹 제약을 지키면서 라벨 비율도 최대한 균등하게 맞춘다.


In [12]:
from sklearn.model_selection import StratifiedGroupKFold

N_SPLITS = 5
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

train_df['fold'] = -1
for fold, (_, valid_idx) in enumerate(
    sgkf.split(train_df, train_df['이상거래여부'], groups=train_df['카드KEY'])
):
    train_df.loc[valid_idx, 'fold'] = fold

assert (train_df['fold'] >= 0).all(), "fold가 배정되지 않은 행이 있습니다."
print("fold별 행 수:")
print(train_df['fold'].value_counts().sort_index())

fold별 행 수:
fold
0    347140
1    347309
2    347138
3    347111
4    347187
Name: count, dtype: int64


In [13]:
# (1) 그룹 무결성: 같은 카드가 서로 다른 fold에 걸쳐 있으면 안 된다
card_fold_counts = train_df.groupby('카드KEY')['fold'].nunique()
n_leaked_cards = (card_fold_counts > 1).sum()
print(f"2개 이상 fold에 걸쳐 있는 카드 수: {n_leaked_cards} (0이어야 정상)")
assert n_leaked_cards == 0, "카드가 여러 fold에 걸쳐 있습니다 — 그룹 제약이 깨졌습니다."

# (2) 층화 품질: fold별 이상거래 비율이 전체 비율(3.687%)과 비슷한지 확인
fold_fraud_rate = train_df.groupby('fold')['이상거래여부'].mean() * 100
overall_rate = train_df['이상거래여부'].mean() * 100
print(f"\nfold별 이상거래 비율(%) (전체 대비 {overall_rate:.3f}%):")
print(fold_fraud_rate.round(3))

# 사용 예시: 특정 fold를 검증셋으로 쓰려면
# f = 0
# tr2 = train_df[train_df['fold'] != f]
# va = train_df[train_df['fold'] == f]

2개 이상 fold에 걸쳐 있는 카드 수: 0 (0이어야 정상)

fold별 이상거래 비율(%) (전체 대비 3.687%):
fold
0    3.683
1    3.701
2    3.682
3    3.683
4    3.688
Name: 이상거래여부, dtype: float64


## 10. 저장

In [14]:
# 이 환경에는 parquet 엔진(pyarrow/fastparquet)이 정상 설치되지 않아(컴파일된 확장 모듈 DLL 로드 실패) CSV로 저장한다.
# category dtype은 CSV 저장 시 문자열로 풀리므로, 다시 읽을 때는 아래 8절에서 정리한 CATEGORICAL_COLS 목록을
# 기준으로 astype('category')를 다시 적용하면 된다.
train_out = os.path.join(OUT_DIR, 'train_processed.csv')
test_out = os.path.join(OUT_DIR, 'test_processed.csv')

train_df.to_csv(train_out, index=False, encoding='utf-8')
test_df.to_csv(test_out, index=False, encoding='utf-8')

print(f"저장 완료: {train_out}  ({os.path.getsize(train_out)/1024/1024:.1f} MB)")
print(f"저장 완료: {test_out}  ({os.path.getsize(test_out)/1024/1024:.1f} MB)")

# 피처 Tier 정의도 함께 저장해, 이후 모델링 노트북이 이 전처리를 다시 실행하지 않고도
# select_features()와 동일한 정보를 로드해서 쓸 수 있게 한다.
tiers_out = os.path.join(OUT_DIR, 'feature_tiers.json')
with open(tiers_out, 'w', encoding='utf-8') as f:
    json.dump(FEATURE_TIERS, f, ensure_ascii=False, indent=2)
print(f"저장 완료: {tiers_out}")


저장 완료: ../.data\processed\train_processed.csv  (494.4 MB)
저장 완료: ../.data\processed\test_processed.csv  (61.4 MB)
저장 완료: ../.data\processed\feature_tiers.json


## 요약

**최종 사용 피처 (42개, 8절 4-Tier 기준 — 프로덕션 스키마 대응력 순)**

| Tier | 개수 | 컬럼 | 프로덕션 가용성 |
|---|---|---|---|
| Tier 0 (transactions 핵심필드만으로 계산) | 16개 | 승인시간대, 통합승인금액, 거래일자\*, 거래연월\*, 거래요일_한글, 시간대구간, 월말여부, 취소성거래_추정, 최근7일사용횟수, 카드누적사용액, 사용자평균사용액_확장, 사용자표준편차_확장, 거래금액_Zscore_확장, 카드첫거래여부, 가맹점평균금액_확장, 가맹점첫거래여부 | 거의 확실히 유지됨 |
| Tier 1 (가맹점 속성 원본) | 7개 | 가맹점광역시도코드, 가맹점승인업종코드, 가맹점형태구분코드, 가맹점상태코드, 가맹점여부_신규, 가맹점누적매출금액_구간화, 인터넷판매여부 | 값 자체는 못 씀 — `merchant_categories` 캐시(§7-1)로 대체, 스키마 다름 |
| Tier 2 (조직 데이터) | 0개 | (placeholder — 부서/직급 등, EDA "추가 확보하면 좋은 데이터") | 이 데이터셋엔 없음, `users`/`teams` 조인 필요 |
| Tier 3 (카드사 정산 필드) | 19개 | 개인법인구분코드_회원, 국내해외여부, 카드이용한도금액, 카드이용한도금액_구간, 법인카드_한도미기재추정, 연령, 남녀구분코드, 승인거래코드, 승인발생경로코드, 개인법인구분코드_가맹점, 로고구분코드, 일시불할부구분코드, 카드구분코드, 할부가능개월수, 전월_매출건수, 전월_매출금액, 전월매출건수_음수여부, 전월매출금액_음수여부, 경과일수_최종이용일자 | `raw_payload` 포함 여부 미확정(§11.2) |

\* `거래일자`(datetime)·`거래연월`(Period)는 dtype이 그대로라 모델에 바로 넣으면 에러가 난다 — 모델링 단계에서 숫자로 변환하거나 제외해야 한다.

`select_features([...])`로 Tier 조합을 골라 쓸 수 있고, `FEATURE_TIERS`는 `.data/processed/feature_tiers.json`으로도 저장했다.

**제외한 컬럼** (`feature_cols`에 없음)
- ID: `카드KEY`, `가맹점KEY`, `승인SEQ`
- 라벨/설명: `이상거래여부`, `이상거래유형`, `이상거래설명` (EDA 10-3절: 카드사 라벨 — 타깃 후보로만 별도 취급, 그대로 피처화 금지)
- 원본 날짜 코드: `기준년월`, `승인일자` (파생 변수로 대체)
- `fold`: 9절에서 만든 검증용 컬럼, 피처 아님

**이 노트북에서 한 일**
1. `train`/`test`가 같은 모집단의 무작위 분할임을 확인하고, 시간 기반 피처는 두 데이터를 합쳐서 계산한 뒤 재분리했다.
2. `'_'` 플레이스홀더를 명시적 NaN으로 정규화했다.
3. 날짜·시간대 파생 변수를 추가했다(거래일자/거래연월/거래요일/시간대구간/월말여부).
4. 0원 거래·전월 실적 음수·법인카드 한도 미기재를 **삭제 대신 플래그**로 남겼다.
5. Look-ahead 피처(EDA 9절 버전)는 만들지 않고, Expanding Window 기반 leak-free 피처(EDA 9-1절 버전)만 생성했다.
6. 콜드스타트(첫 거래) 상태를 `fillna`로 지우지 않고 플래그+결측으로 보존했다(EDA 10-4절 교훈).
7. 범주형 변수를 `category` dtype으로 정리하고(원-핫 인코딩 없음), 원본 컬럼은 삭제 없이 모두 보존했다.
8. `train_df`에 카드 단위 `StratifiedGroupKFold` 기반 `fold` 컬럼을 추가해, 모델 개발 시 test를 건드리지 않고도 검증할 수 있게 했다.
9. 피처를 프로덕션 스키마(`transactions` 핵심필드/가맹점 캐시/조직 데이터/카드사 정산필드) 기준 4-Tier로 재구성해, raw_payload 확정 전에도 `select_features()`로 안전한 조합만 골라 쓸 수 있게 했다.

**의도적으로 하지 않은 것**
- **스케일링**: EDA 10절에서 권장한 트리 기반 앙상블(LightGBM/XGBoost)은 스케일링이 필요 없다. 선형/거리 기반 모델을
  쓴다면 이 노트북 이후 단계에서 `feature_cols` 중 수치형만 골라 별도로 스케일링해야 한다.
- **결측치 임퓨테이션**: 트리 기반 모델은 결측을 네이티브로 처리할 수 있고, 임의로 채우면 콜드스타트 신호가 사라진다
  (EDA 10-4절). 결측을 반드시 채워야 하는 모델을 쓸 경우, 최소한 `카드/가맹점 첫거래여부` 플래그는 함께 넣어야 한다.
- **`이상거래여부`를 곧바로 지도학습 타깃으로 확정하는 것**: EDA 10-3절 결론대로 이 라벨은 카드사 부정사용 탐지
  기준이지 정산 담당자의 승인/반려 기준이 아니다. 이 노트북은 라벨을 보존만 하며, 실제 타깃 확정은 모델링 단계의
  판단으로 남겨둔다.